# Other

In [13]:
import pandas as pd
import re
from bs4 import BeautifulSoup

In [14]:
pd.set_option("display.max_rows", None)       # показывать все строки
pd.set_option("display.max_columns", None)    # показывать все столбцы
pd.set_option("display.max_colwidth", None)   # не обрезать содержимое ячеек
pd.set_option("display.width", None)          # автоматически определять ширину вывода

In [15]:
data = pd.read_feather("../data/raw/candidate_data/articles.f")

d:\Работа\Avito\Тестовые задания\AvitoDataScienceBootCampTestTask\.venv\Lib\site-packages\pandas\io\feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [16]:
def html_to_words(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    # Удаляем содержимое служебных тегов
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Получаем обычный текст
    text = soup.get_text(separator=" ", strip=True)

    # Приводим к нижнему регистру
    text = text.lower()

    # Оставляем слова и числа
    words = re.findall(r"[а-яёa-z0-9]+", text)

    return " ".join(words)

In [17]:
data["prepare_body"] = data["body"].apply(lambda x: html_to_words(x))

# QDRANT

In [18]:
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding, TextEmbedding

In [19]:
QDRANT_URL = "http://127.0.0.1:6333"
COLLECTION_NAME = "russian_hybrid"

DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "bm25"

DENSE_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
SPARSE_MODEL_NAME = "Qdrant/bm25"

In [20]:
client = QdrantClient(url=QDRANT_URL, timeout=30.0)

In [21]:
dense_model = TextEmbedding(
    model_name=DENSE_MODEL_NAME,
)

sparse_model = SparseTextEmbedding(
    model_name=SPARSE_MODEL_NAME,

    # Русские стоп-слова и Snowball stemmer для русского языка
    language="russian",

    # Стандартные параметры BM25
    k=1.2,
    b=0.75,

    # Желательно подобрать под среднюю длину документов в корпусе
    avg_len=256.0,
)

C:\Users\Даниил\AppData\Local\Temp\ipykernel_34448\4170479625.py:1: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding(


In [22]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "title_dense": models.VectorParams(
                size=dense_model.embedding_size,
                distance=models.Distance.COSINE
            ),
            "body_dense": models.VectorParams(
                size=dense_model.embedding_size,
                distance=models.Distance.COSINE
            ),
        },
        sparse_vectors_config={
            "title_sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False
                ),
                modifier=models.Modifier.IDF
            ),
            "body_sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False
                ),
                modifier=models.Modifier.IDF
            )
        }
    )

In [23]:
def get_dense_vectors(texts):
    return list(dense_model.passage_embed(texts))

def get_sparse_vectors(texts):
    return list(sparse_model.embed(texts))

def load_data_in_qdrant(client, data: pd.DataFrame):
    artilce_ids = data["article_id"].to_list()
    titles = data["title"].to_list()
    bodies = data["prepare_body"].to_list()

    title_dense_vectors = get_dense_vectors(titles)
    body_dense_vectors = get_dense_vectors(bodies)

    title_sparse_vectors = get_sparse_vectors(titles)
    body_sparse_vectors = get_sparse_vectors(bodies)


    points: list[models.PointStruct] = []

    for (
        article_id,
        title,
        body,
        title_dense,
        body_dense,
        title_sparse,
        body_sparse
    ) in zip(
        artilce_ids,
        titles,
        bodies,
        title_dense_vectors,
        body_dense_vectors,
        title_sparse_vectors,
        body_sparse_vectors
    ):
        points.append(
            models.PointStruct(
                id=article_id,
                vector={
                    "title_dense": title_dense.tolist(),

                    "body_dense": body_dense.tolist(),

                    "title_sparse": models.SparseVector(
                        indices=title_sparse.indices.tolist(),
                        values=title_sparse.values.tolist()
                    ),

                    "body_sparse": models.SparseVector(
                        indices=body_sparse.indices.tolist(),
                        values=body_sparse.values.tolist()
                    )
                },
                payload={
                    "title": title,
                    "body": body
                }
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True
    )

In [24]:
load_data_in_qdrant(client, data)

In [ ]:
def hybrid_article_search(
    query: str,
    limit: int = 10,
    prefetch_limit: int = 50
):
    dense_query = next(
        dense_model.query_embed(query)
    )

    sparse_query = next(
        sparse_model.query_embed(query)
    )

    sparse_vector = models.SparseVector(
        indices=sparse_query.indices.tolist(),
        values=sparse_query.values.tolist()
    )

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            # Поиск по заголовкам
            models.Prefetch(
                query=sparse_vector,
                using="title_sparse",
                limit=prefetch_limit
            ),

            # Поиск по текстам статей
            models.Prefetch(
                query=sparse_vector,
                using="body_sparse",
                limit=prefetch_limit
            ),

            # Семантический поиск
            models.Prefetch(
                query=dense_query.tolist(),
                using="title_dense",
                limit=prefetch_limit
            ),

            models.Prefetch(
                query=dense_query.tolist(),
                using="body_dense",
                limit=prefetch_limit
            )
        ],

        # Заголовок имеет больший вес
        query=models.RrfQuery(
            rrf=models.Rrf(
                weights=[
                    2.0,  # title_sparse
                    1.0,  # body_sparse
                    2.0,  # title_dense
                    1.0   # body_dense
                ]
            )
        ),

        limit=limit,
        with_payload=True
    )

    return response.points

In [26]:
queries = pd.read_feather("../data/raw/candidate_data/calibration.f")

d:\Работа\Avito\Тестовые задания\AvitoDataScienceBootCampTestTask\.venv\Lib\site-packages\pandas\io\feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [27]:
queries.head()

,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито доставкой и оплатить также через авито, в случае возврата товара деньги безусловно мне возвращаются?",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909
3,4,как получить деньги за возрат если продавец уже забрал товар?,4400 4403
4,5,"Когда мне прийдут деньги за доставку, сегодня уже <DATE>",4361


In [33]:
[point.id for point in hybrid_article_search("как получить деньги за возрат если продавец уже забрал товар?")]

[2698, 4403, 4400, 2037, 4308, 2866, 2696, 4361, 3169, 2865]